<a href="https://colab.research.google.com/github/busraparlakk/Auto-MPG/blob/main/gemma4_finetune_duzeltilmis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma-4 Türkçe Q&A Fine-Tuning (Düzeltilmiş)



In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --upgrade torchao
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
!pip install --no-deps --upgrade timm

## 1. Modeli yükle

In [2]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit",
    dtype = None,                # otomatik tespit
    max_seq_length = 128,        # 128'den artırıldı
    load_in_4bit = True,
    full_finetuning = False,
    # token = "YOUR_HF_TOKEN",   # gated modeller için
)

==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.8.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

## 2. Fine-tuning öncesi inference (baseline)

In [ ]:
from transformers import TextStreamer

def do_gemma_4_inference(messages, max_new_tokens = 256):
    _ = model.generate(
        **tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True,
            tokenize = True,
            return_dict = True,
            return_tensors = "pt",
        ).to("cuda"),
        max_new_tokens = max_new_tokens,
        temperature = 0.3, top_p = 0.95, top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True),
        use_cache = True
    )

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type": "text",
                 "text": "Madde 23'ün hangi tarihte ve hangi Resmî Gazete sayısında yayımlandığı orijinal yönetmelikte belirtilmiştir?"}]
}]
do_gemma_4_inference(messages, max_new_tokens = 256)

Lütfen bahsettiğiniz **yönetmeliğin adını** belirtin. Hangi yönetmelikten bahsettiğinizi bilmeden, Madde 23'ün hangi tarihte ve hangi Resmî Gazete sayısında yayımlandığını söyleyemem.<turn|>


## 3. LoRA adaptörlerini ekle

LoRA r=8 (mevcut ayar korundu — hafif ve hızlı). Daha güçlü bir model isterseniz r=16 deneyebilirsiniz.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # sadece metin
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

## 4. Gemma-4 chat template'ini uygula

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

## 5. Google Drive'ı bağla ve veri setlerini yükle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/BTK/Day3/anayasa

/content/drive/MyDrive/BTK/Day3/anayasa


In [ ]:
# Ayrı train.csv ve test.csv dosyalarını yükle
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

print(f"Train örnek sayısı: {len(train_df)}")
print(f"Test  örnek sayısı: {len(test_df)}")
print(f"\nTrain DataFrame kolonları: {list(train_df.columns)}")
print(f"\nİlk train örneği:")
print(train_df.head(1))

Train örnek sayısı: 4350
Test  örnek sayısı: 1107

Train DataFrame kolonları: ['question', 'answer']

İlk train örneği:
                                            question  \
0  Devlet yetkilerinin kullanılması hangi ilkeler...   

                                              answer  
0  Devlet yetkilerinin kullanılması Anayasaya ve ...  


In [ ]:
from datasets import Dataset

# DİKKAT: Dataset.from_pandas'ta `split` argümanı dilimleme yapmaz, sadece bir etikettir.
# Bu yüzden `df.head(N)` veya `.train_test_split(...)` kullanılmalı.
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
eval_dataset  = Dataset.from_pandas(test_df,  preserve_index=False)

print(f"train_dataset uzunluğu: {len(train_dataset)}")
print(f"eval_dataset  uzunluğu: {len(eval_dataset)}")
print(f"\nÖrnek kayıt:")
print(train_dataset[0])

train_dataset uzunluğu: 4350
eval_dataset  uzunluğu: 1107

Örnek kayıt:
{'question': 'Devlet yetkilerinin kullanılması hangi ilkelere bağlıdır?', 'answer': 'Devlet yetkilerinin kullanılması Anayasaya ve kanunlara bağlıdır. Cumhurbaşkanı yürütme yetkisini, mahkemeler yargı yetkisini ve Millet Meclisi yasama yetkisini bu ilkelere uygun olarak kullanmalıdır.'}


## 6. Q&A çiftlerini chat template formatına çevir

Her satırı `<start_of_turn>user ... <start_of_turn>model ...` yapısında tek bir `text` kolonuna dönüştürüyoruz.

In [ ]:
def create_text_column(example):
    user_message      = example["question"]
    assistant_message = example["answer"]
    messages = [
        {"role": "user",      "content": user_message},
        {"role": "assistant", "content": assistant_message},
    ]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = False,  # eğitim için False olmalı
    )
    return {"text": formatted_text}

train_dataset = train_dataset.map(create_text_column, remove_columns=["question", "answer"])
eval_dataset  = eval_dataset.map(create_text_column,  remove_columns=["question", "answer"])

print("Formatlanmış ilk train örneği:\n")
print(train_dataset[0]["text"])

Map:   0%|          | 0/4350 [00:00<?, ? examples/s]

Map:   0%|          | 0/1107 [00:00<?, ? examples/s]

Formatlanmış ilk train örneği:

<bos><|turn>user
Devlet yetkilerinin kullanılması hangi ilkelere bağlıdır?<turn|>
<|turn>model
Devlet yetkilerinin kullanılması Anayasaya ve kanunlara bağlıdır. Cumhurbaşkanı yürütme yetkisini, mahkemeler yargı yetkisini ve Millet Meclisi yasama yetkisini bu ilkelere uygun olarak kullanmalıdır.<turn|>



## 7. SFTTrainer'ı kur

- `num_train_epochs = 2` → 2 tam epoch eğitim (max_steps yerine)
- `eval_strategy = "steps"` + `eval_steps = 50` → her 50 adımda eval loss'u logla
- `dataset_text_field = "text"` → formatlanmış kolonu işaret et

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        per_device_eval_batch_size  = 1,
        gradient_accumulation_steps = 4,         # effective batch size = 4
        warmup_steps = 5,
        num_train_epochs = 1,                    # max_steps yerine epoch bazlı
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "steps",                 # eval'i de takip et
        eval_steps = 50,
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 2,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/4350 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1107 [00:00<?, ? examples/s]

## 8. Sadece cevap (response) üzerinden loss hesapla

`train_on_responses_only` kullanıcı sorusunu maskeleyip sadece asistan cevabı üzerinden gradient akmasını sağlar. Bu, modelin soru üretmek yerine cevap vermeyi öğrenmesi için kritik.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part    = "<|turn>model\n",
)

Map (num_proc=16):   0%|          | 0/4350 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/4350 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/1107 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/1107 [00:00<?, ? examples/s]

## 9. Maskeleme doğrulaması (ÖNEMLİ)

Aşağıdaki iki çıktıyı **dikkatlice karşılaştır**:
- Birinci çıktı: tüm girdi (soru + cevap)
- İkinci çıktı: sadece **cevap kısmı** görünmeli, soru kısmı `<pad>` ile maskelenmiş olmalı

Eğer ikinci çıktıda da soru görünüyorsa, `instruction_part` / `response_part` token'ları yanlıştır ve maskeleme çalışmıyor demektir.

In [ ]:
print(repr(train_dataset[0]["text"]))

'<bos><|turn>user\nDevlet yetkilerinin kullanılması hangi ilkelere bağlıdır?<turn|>\n<|turn>model\nDevlet yetkilerinin kullanılması Anayasaya ve kanunlara bağlıdır. Cumhurbaşkanı yürütme yetkisini, mahkemeler yargı yetkisini ve Millet Meclisi yasama yetkisini bu ilkelere uygun olarak kullanmalıdır.<turn|>\n'


In [ ]:
# Tüm girdi (input_ids decode edilmiş)
print("===== INPUT_IDS (tam girdi) =====")
print(tokenizer.decode(trainer.train_dataset[0]["input_ids"]))

===== INPUT_IDS (tam girdi) =====
<bos><|turn>user
Devlet yetkilerinin kullanılması hangi ilkelere bağlıdır?<turn|>
<|turn>model
Devlet yetkilerinin kullanılması Anayasaya ve kanunlara bağlıdır. Cumhurbaşkanı yürütme yetkisini, mahkemeler yargı yetkisini ve Millet Meclisi yasama yetkisini bu ilkelere uygun olarak kullanmalıdır.<turn|>



In [ ]:
# Sadece labels (loss hesaplanan kısım) — soru maskelenmiş olmalı
print("===== LABELS (sadece cevap görünmeli) =====")
print(tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]
).replace(tokenizer.pad_token, " "))

===== LABELS (sadece cevap görünmeli) =====
                       Devlet yetkilerinin kullanılması Anayasaya ve kanunlara bağlıdır. Cumhurbaşkanı yürütme yetkisini, mahkemeler yargı yetkisini ve Millet Meclisi yasama yetkisini bu ilkelere uygun olarak kullanmalıdır.<turn|>



## 10. GPU bellek durumu (eğitim öncesi)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.494 GB.
7.672 GB of memory reserved.


## 11. Eğitimi başlat

~4350 train örneği × 2 epoch / (batch_size 1 × grad_accum 4) ≈ **2175 step**.
Colab Pro T4'te tahminen 1-2 saat sürer. Eval loss train loss ile birlikte düşmeli; eval loss yükselmeye başlarsa overfit oluyordur.

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,350 | Num Epochs = 2 | Total steps = 2,176
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 12,668,928 of 5,135,846,944 (0.25% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss,Validation Loss
50,0.961839,2.514870
100,0.689354,2.189013
150,0.614598,2.034487
200,0.524224,1.992523
250,0.483504,1.884692
300,0.485984,1.913777
350,0.482970,1.880513
400,0.459524,1.838148
450,0.463778,1.813742
500,0.446272,1.775726


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-800/token

## 12. Eğitim sonrası istatistikler

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

16939.6133 seconds used for training.
282.33 minutes used for training.
Peak reserved memory = 7.916 GB.
Peak reserved memory for training = 0.244 GB.
Peak reserved memory % of max memory = 20.044 %.
Peak reserved memory for training % of max memory = 0.618 %.


## 13. Fine-tune edilmiş modelle test

In [ ]:
messages = [{
    "role": "user",
    "content": [{
        "type": "text",
        "text": "Madde 23 hangi konuları düzenlemektedir?",
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    temperature = 0.3, top_p = 0.95, top_k = 64,do_sample = True,
)
print(tokenizer.batch_decode(outputs)[0])
#Madde 23, komisyonlara havale, esas ve tali komisyonların tanımı ve işleyişi konularını düzenlemektedir. Tekliflerin ve Cumhurbaşkanlığı kararnamelerinin komisyonlara nasıl gönderileceğini ve komisyonların rollerini belirler.

<bos><|turn>user
Madde 23 hangi konuları düzenlemektedir?<turn|>
<|turn>model
Madde 23, Türkiye Büyük Millet Meclisi'nin çalışma usul ve esaslarını, komisyonların işleyişini, tutanakların tutulmasını, işaret ve ad çalınmasını önlemek için alınan tedbirleri ve Meclis'in genel işleyişini düzenlemektedir.<turn|>


In [ ]:
# Modelin gerçekten LoRA adaptörlerine sahip olduğunu doğrula
print(type(model).__name__)  # PeftModel veya benzeri görmelisin
print(model.peft_config)     # LoRA config'i listelemeli

PeftModelForCausalLM
{'default': LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping={'base_model_class': 'Gemma4ForConditionalGeneration', 'parent_library': 'transformers.models.gemma4.modeling_gemma4', 'unsloth_fixed': True}, peft_version='0.19.1', base_model_name_or_path='unsloth/gemma-4-E2B-it-unsloth-bnb-4bit', revision=None, inference_mode=False, r=8, target_modules='(?:.*?(?:language|text).*?(?:self_attn|attention|attn|mlp|feed_forward|ffn|dense).*?(?:k_proj|q_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|per_layer_input_gate|per_layer_projection|linear|embedding_projection|relative_k_proj).*?)|(?:\\bmodel\\.layers\\.[\\d]{1,}\\.(?:self_attn|attention|attn|mlp|feed_forward|ffn|dense)\\.(?:(?:k_proj|q_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|per_layer_input_gate|per_layer_projection|linear|embedding_projection|relative_k_proj)))', exclude_modules=None, lora_alpha=8, lora_dropout=0, fan_in_fan_out=False, bias='none', use_r

In [ ]:
# 1. Eğitim verisinden bir soru (model bunu görmüş olmalı)
test_questions = [
    "Madde 13'te neden 'ancak kanunla' sınırlanabileceği belirtilmiştir?",
    # 2. Veride olmayan ama domain'e uygun bir soru
    "Türkiye Cumhuriyeti'nin yönetim biçimi nedir?",
    # 3. Tamamen alakasız (model bunu da makul cevaplamalı, ezberlememeli)
    "Python'da liste nasıl oluşturulur?",
]

for q in test_questions:
    print(f"\n{'='*70}\nSORU: {q}\n{'='*70}")
    messages = [{"role": "user", "content": [{"type": "text", "text": q}]}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", tokenize=True, return_dict=True,
    ).to("cuda")
    outputs = model.generate(
        **inputs, max_new_tokens=300,
        temperature=0.3, top_p=0.95, do_sample=True,
    )
    # Sadece yeni üretilen kısmı göster
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(response)


SORU: Madde 13'te neden 'ancak kanunla' sınırlanabileceği belirtilmiştir?
Madde 13'te 'ancak kanunla' sınırlanabileceği belirtilmiştir çünkü temel hak ve hürriyetler anayasal olarak güvence altına alınmıştır. Bu haklar, keyfi olarak sınırlanamaz; yalnızca kanunla ve belirli koşullar altında sınırlanabilirler. Bu, bireysel hakların mutlaklığını korumak için kritik bir ilkedir.

SORU: Türkiye Cumhuriyeti'nin yönetim biçimi nedir?
Türkiye Cumhuriyeti, cumhuriyet yönetim biçimini benimsemiştir. Bu yönetim şekli, egemenliğin kayıtsız şartsız millette olduğunu, devletin halk tarafından temsil edildiğini ve egemenliğin tek kaynağı olan millet olduğunu ifade eder.

SORU: Python'da liste nasıl oluşturulur?
Python'da liste, köşeli parantezler `[]` kullanılarak oluşturulur ve içindeki elemanlar virgülle ayrılır.

**Örnekler:**

1. **Boş liste:**
   ```python
   liste1 = []
   print(liste1)  # Çıktı: []
   ```

2. **Sayı içeren liste:**
   ```python
   sayilar = [1, 5, 10, 15, 20]
   print(sayila

## 14. LoRA adaptörlerini kaydet

In [ ]:
model.save_pretrained("gemma_4_lora")
tokenizer.save_pretrained("gemma_4_lora")

# HuggingFace Hub'a yüklemek için:
# model.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN")
# tokenizer.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN")

Unsloth: Restored added_tokens_decoder metadata in gemma_4_lora/tokenizer_config.json.


['gemma_4_lora/processor_config.json']

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

model.push_to_hub("maydogan/gemma4-turkish-sorucevap", token=userdata.get('HF_TOKEN'))
tokenizer.push_to_hub("maydogan/gemma4-turkish-sorucevap", token=userdata.get('HF_TOKEN'))

print("Push tamamlandı!")
print("Model adresi: https://huggingface.co/maydogan/gemma4-turkish-sorucevap")

README.md:   0%|          | 0.00/570 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  565kB / 50.7MB            

Saved model to https://huggingface.co/maydogan/gemma4-turkish-sorucevap


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpv3npuvzj/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpv3npuvzj/tokenizer.json: 100%|##########| 32.2MB / 32.2MB            

Push tamamlandı!
Model adresi: https://huggingface.co/maydogan/gemma4-turkish-sorucevap


**KULLANIM**

In [1]:
# Hücre 1 — Kurulum
%%capture
!pip install unsloth
!pip install --no-deps --upgrade transformers

# Hücre 2 — Modeli yükle
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = "maydogan/gemma4-turkish-sorucevap",
    max_seq_length = 512,
    load_in_4bit = True,
)
FastModel.for_inference(model)
print("Model hazır!")



In [5]:
# Hücre 3 — Soru sor
soru = "eşekk demek suç mu"  # ← buraya yaz

messages = [{"role": "user", "content": [{"type": "text", "text": soru}]}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", tokenize=True, return_dict=True,
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 300,
    temperature = 0.3,
    do_sample = True,
)
cevap = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"\n{'='*60}\nCEVAP:\n{'='*60}\n{cevap}")


CEVAP:
"Eşekk" kelimesinin tek başına bir suç teşkil edip etmediği, **hangi bağlamda ve hangi yasal düzenlemelere göre değerlendirildiğine bağlıdır.**

Bu kelime, genellikle **argo veya hakaret içeren bir ifade** olarak kullanılabilir. Eğer bu ifade, bir kişiye yönelik **hakaret, tehdit, aşağılama veya taciz** amacıyla kullanılıyorsa, bu durum **ceza hukuku kapsamında suç teşkil edebilir.**

**Özetle:**

1. **Sadece kelimeyi söylemek:** Kelimenin kendisi suç değildir.
2. **Kullanım şekli:** Eğer bu ifade, başkasına zarar verme, onur kırıcı bir davranış sergileme veya kanunları ihlal etme amacı taşıyorsa, **suç olabilir.**

**Hukuki bir durum söz konusuysa, kesin bir yargıya varmak için:**

* **İlgili ceza kanunlarını** incelemek gerekir.
* Durumun **somut delillerle** değerlendirilmesi gerekir.
* Bir **hukuk uzmanına** danışmak en doğru yol olacaktır.


## 15. (Opsiyonel) Kaydedilen LoRA'yı yeniden yükle

In [ ]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "gemma_4_lora",
        max_seq_length = 512,
        load_in_4bit = True,
    )

    messages = [{
        "role": "user",
        "content": [{"type": "text", "text": "Hırsızlık suçu nedir"}]
    }]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
        tokenize = True,
        return_dict = True,
    ).to("cuda")

    from transformers import TextStreamer
    _ = model.generate(
        **inputs,
        max_new_tokens = 512,
        temperature = 1.0, top_p = 0.95, top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True),
    )

## 16. (Opsiyonel) Birleştirilmiş modeli HuggingFace'e yükle

In [ ]:
if False:
    from google.colab import userdata
    model.push_to_hub_merged(
        "maydogan/gemma-4-finetune", tokenizer,
        token = userdata.get('HF_TOKEN')
    )